In [ ]:
"""
Assignment 4 — Kalman Filter and Parameter Estimation
02417 Time Series Analysis

This script generates all 13 plots for the assignment report.
It can also be converted to a Jupyter notebook via:
    jupytext --to notebook assignment4_notebook.py
"""

# Assignment 4 — Kalman Filter and Parameter Estimation
## 02417 Time Series Analysis

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from scipy.stats import t as t_dist, norm
import os, warnings
warnings.filterwarnings("ignore")

plt.rcParams.update({
    "figure.dpi": 150,
    "figure.figsize": (10, 5),
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 11,
})

out_dir = "plots"
os.makedirs(out_dir, exist_ok=True)

def save_fig(name, fig=None):
    if fig is None:
        fig = plt.gcf()
    fig.savefig(os.path.join(out_dir, f"{name}.png"),
                dpi=200, bbox_inches="tight", facecolor="white")
    plt.show()

---
# Part 1 — Parameter Estimation in a Simple State-Space Model

$$X_t = a\,X_{t-1} + b + e_{1,t}, \quad e_{1,t}\sim\mathcal{N}(0,\sigma_1^2)$$
$$Y_t = X_t + e_{2,t}, \quad e_{2,t}\sim\mathcal{N}(0,\sigma_2^2)$$

## 1.1 — Five independent realisations

In [ ]:
np.random.seed(42)

def simulate_process(n, a, b, sigma1, X0):
    """Simulate X_t = a*X_{t-1} + b + e1 for t=1..n."""
    X = np.zeros(n + 1)
    X[0] = X0
    for t_ in range(1, n + 1):
        X[t_] = a * X[t_ - 1] + b + np.random.normal(0, sigma1)
    return X[1:]  # X_1 ... X_n

n, a, b, sigma1, X0 = 100, 0.9, 1.0, 1.0, 5.0

fig, ax = plt.subplots(figsize=(10, 5))
for i in range(5):
    X = simulate_process(n, a, b, sigma1, X0)
    ax.plot(np.arange(1, n + 1), X, linewidth=0.8, label=f"Realisation {i+1}")
ax.set_xlabel("Time index t")
ax.set_ylabel(r"$X_t$")
ax.set_title("Five Independent Realisations of the State Process")
ax.legend(fontsize=9)
save_fig("image1")

## 1.2 — Single realisation with noisy observations

In [ ]:
np.random.seed(123)
sigma2 = 1.0

X_true = simulate_process(n, a, b, sigma1, X0)
Y_obs = X_true + np.random.normal(0, sigma2, n)
t_idx = np.arange(1, n + 1)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(t_idx, X_true, "k-", linewidth=0.9, label=r"True state $X_t$")
ax.plot(t_idx, Y_obs, "o", color="firebrick", markersize=3.5,
        alpha=0.8, label=r"Observation $Y_t$")
ax.set_xlabel("Time index t")
ax.set_ylabel("Value")
ax.set_title("Latent State and Noisy Observations")
ax.legend()
save_fig("image2")

## 1.3 — Kalman Filter

In [ ]:
def kalman_filter(y, theta, R, x_prior=0.0, P_prior=10.0):
    """
    Scalar Kalman filter for X_t = a*X_{t-1} + b + e1, Y_t = X_t + e2.
    theta = (a, b, sigma1).
    Returns dict with predicted/filtered means & variances, innovations.
    """
    a_, b_, s1 = theta
    N = len(y)
    x_pred = np.zeros(N)
    P_pred = np.zeros(N)
    x_filt = np.zeros(N)
    P_filt = np.zeros(N)
    innov = np.zeros(N)
    innov_var = np.zeros(N)

    for t_ in range(N):
        # prediction
        if t_ == 0:
            x_pred[t_] = a_ * x_prior + b_
            P_pred[t_] = a_**2 * P_prior + s1**2
        else:
            x_pred[t_] = a_ * x_filt[t_ - 1] + b_
            P_pred[t_] = a_**2 * P_filt[t_ - 1] + s1**2

        # update
        innov[t_] = y[t_] - x_pred[t_]
        innov_var[t_] = P_pred[t_] + R
        K = P_pred[t_] / innov_var[t_]
        x_filt[t_] = x_pred[t_] + K * innov[t_]
        P_filt[t_] = (1 - K) * P_pred[t_]

    return dict(x_pred=x_pred, P_pred=P_pred,
                x_filt=x_filt, P_filt=P_filt,
                innov=innov, innov_var=innov_var)

# Run with true parameters
kf = kalman_filter(Y_obs, theta=(a, b, sigma1), R=sigma2**2,
                   x_prior=X0, P_prior=1.0)

lo = kf["x_pred"] - 1.96 * np.sqrt(kf["P_pred"])
hi = kf["x_pred"] + 1.96 * np.sqrt(kf["P_pred"])

fig, ax = plt.subplots(figsize=(10, 5))
ax.fill_between(t_idx, lo, hi, alpha=0.2, color="steelblue", label="95% CI")
ax.plot(t_idx, X_true, "k-", linewidth=0.8, label=r"True state $X_t$")
ax.plot(t_idx, Y_obs, "o", color="firebrick", markersize=3, alpha=0.7,
        label=r"Observation $Y_t$")
ax.plot(t_idx, kf["x_pred"], "--", color="steelblue", linewidth=0.8,
        label="Predicted state")
ax.set_xlabel("Time index t")
ax.set_ylabel("Value")
ax.set_title("Kalman Filter: One-Step Prediction with 95% Confidence Interval")
ax.legend(fontsize=9)
save_fig("image3")

coverage = np.mean((X_true >= lo) & (X_true <= hi))
print(f"Coverage: {coverage:.2f}")

## 1.4 — Maximum-likelihood parameter estimation

In [ ]:
def neg_loglik(theta, y, R, x_prior=0.0, P_prior=10.0):
    """Negative log-likelihood for minimisation."""
    if theta[2] <= 0:
        return 1e10
    kf = kalman_filter(y, theta, R, x_prior, P_prior)
    e = kf["innov"]
    S = kf["innov_var"]
    ll = -0.5 * np.sum(np.log(2 * np.pi * S) + e**2 / S)
    return -ll

def estimate_params(y, R, x0_init=5.0):
    """Estimate (a, b, sigma1) by MLE with data-driven initialization."""
    # Data-driven starting guess for b: mean(y) * (1 - a_guess)
    y_mean = np.mean(y)
    starts = [
        [0.5, 0.5, 1.0],
        [0.8, y_mean * 0.1, 1.0],
        [0.9, y_mean * 0.1, np.std(np.diff(y))],
    ]
    best_val = np.inf
    best_par = np.full(3, np.nan)
    for s in starts:
        res = minimize(neg_loglik, x0=s,
                       args=(y, R, x0_init, 1.0),
                       method="L-BFGS-B",
                       bounds=[(-0.999, 0.999), (-50, 50), (1e-4, 50)])
        if res.success and res.fun < best_val:
            best_val = res.fun
            best_par = res.x
    return best_par

def run_scenario(a_, b_, s1_, s2_=1.0, n_sim=100, n_obs=100):
    """Run n_sim simulations and return estimates."""
    est = np.zeros((n_sim, 3))
    for i in range(n_sim):
        X = simulate_process(n_obs, a_, b_, s1_, X0=5.0)
        Y = X + np.random.normal(0, s2_, n_obs)
        est[i] = estimate_params(Y, R=s2_**2)
    return est

np.random.seed(2024)
print("Scenario 1 ...")
est_s1 = run_scenario(0.9, 1, 1)
print("Scenario 2 ...")
est_s2 = run_scenario(0.9, 5, 1)
print("Scenario 3 ...")
est_s3 = run_scenario(0.9, 1, 5)

In [ ]:
# Boxplots
par_names = [r"Parameter $a$", r"Parameter $b$", r"Parameter $\sigma_1$"]
true_vals = [
    [0.9, 0.9, 0.9],  # a
    [1.0, 5.0, 1.0],  # b
    [1.0, 1.0, 5.0],  # sigma1
]
labels = ["S1: a=0.9, b=1,\ns1=1",
          "S2: a=0.9, b=5,\ns1=1",
          "S3: a=0.9, b=1,\ns1=5"]

fig, axes = plt.subplots(1, 3, figsize=(13, 5))
for j, ax in enumerate(axes):
    data = [est_s1[:, j], est_s2[:, j], est_s3[:, j]]
    bp = ax.boxplot(data, tick_labels=labels, patch_artist=True,
                    widths=0.5, showfliers=True,
                    flierprops=dict(markersize=3))
    colors = ["#66c2a5", "#fc8d62", "#8da0cb"]
    for patch, c in zip(bp["boxes"], colors):
        patch.set_facecolor(c)
        patch.set_alpha(0.7)
    for k, tv in enumerate(true_vals[j]):
        ax.plot(k + 1, tv, "Dr", markersize=7, zorder=5)
    ax.set_title(par_names[j])
    ax.set_ylabel("Estimated value" if j == 0 else "")
    ax.tick_params(axis="x", labelsize=8)
fig.suptitle("Maximum-Likelihood Parameter Estimates (100 Simulations)", fontsize=13)
fig.tight_layout()
save_fig("image4")

# Print summaries
for name, est, tv in zip(labels, [est_s1, est_s2, est_s3],
                          [(0.9,1,1),(0.9,5,1),(0.9,1,5)]):
    print(f"\n{name.replace(chr(10),' ')}:")
    for j, pn in enumerate(["a","b","sigma1"]):
        vals = est[:, j][~np.isnan(est[:, j])]
        print(f"  {pn}: median={np.median(vals):.3f}, sd={np.std(vals):.3f}")

## 1.5 — System noise from a Student's t-distribution

In [ ]:
# Density comparison
x_grid = np.linspace(-5, 5, 600)
fig, ax = plt.subplots(figsize=(10, 4.5))
ax.plot(x_grid, norm.pdf(x_grid), "k-", linewidth=1, label="Normal")
for nu, c in [(100, "#1b9e77"), (5, "#7570b3"), (2, "#d95f02"), (1, "#e7298a")]:
    ax.plot(x_grid, t_dist.pdf(x_grid, df=nu), color=c, linewidth=1,
            label=fr"$t(\nu={nu})$")
ax.set_xlabel("x")
ax.set_ylabel("Density")
ax.set_title(r"Densities: Standard Normal versus Student's $t$")
ax.legend()
save_fig("image5")

In [ ]:
def simulate_process_t(n, a_, b_, s1_, X0_, nu_):
    """Simulate with t-distributed system noise."""
    X = np.zeros(n + 1)
    X[0] = X0_
    for t_ in range(1, n + 1):
        X[t_] = a_ * X[t_ - 1] + b_ + s1_ * np.random.standard_t(nu_)
    return X[1:]

def run_scenario_t(nu_, n_sim=100, n_obs=100):
    est = np.zeros((n_sim, 3))
    for i in range(n_sim):
        X = simulate_process_t(n_obs, 0.9, 1, 1, 5.0, nu_)
        Y = X + np.random.normal(0, 1.0, n_obs)
        est[i] = estimate_params(Y, R=1.0)
    return est

np.random.seed(7)
nu_vals = [100, 5, 2, 1]
est_t = {}
for nu in nu_vals:
    print(f"nu = {nu} ...")
    est_t[nu] = run_scenario_t(nu)

In [ ]:
# Boxplots
all_labels = ["Gaussian"] + [fr"$t(\nu={v})$" for v in nu_vals]
all_data = [est_s1] + [est_t[v] for v in nu_vals]

fig, axes = plt.subplots(1, 3, figsize=(13, 5))
for j, ax in enumerate(axes):
    data_j = [d[:, j] for d in all_data]
    bp = ax.boxplot(data_j, tick_labels=all_labels, patch_artist=True,
                    widths=0.5, showfliers=True,
                    flierprops=dict(markersize=3))
    colors = ["#66c2a5", "#fc8d62", "#8da0cb", "#e78ac3", "#a6d854"]
    for patch, c in zip(bp["boxes"], colors):
        patch.set_facecolor(c)
        patch.set_alpha(0.7)
    ax.axhline([0.9, 1.0, 1.0][j], color="red", ls="--", lw=1)
    ax.set_title(par_names[j])
    ax.set_ylabel("Estimated value" if j == 0 else "")
    ax.tick_params(axis="x", labelsize=8, rotation=20)
fig.suptitle("Estimates Under Gaussian vs. Heavy-Tailed System Noise", fontsize=13)
fig.tight_layout()
save_fig("image6")

for lbl, d in zip(all_labels, all_data):
    vals = d[~np.isnan(d).any(axis=1)]
    print(f"{lbl}: a_med={np.median(vals[:,0]):.3f}, "
          f"b_med={np.median(vals[:,1]):.3f}, s1_med={np.median(vals[:,2]):.3f}")

---
# Part 2 — Transformer Station Temperature

## 2.1 — Exploratory analysis

In [ ]:
df = pd.read_csv("transformer_data.csv")
print(df.head())
print(f"n = {len(df)}")

var_labels = {
    "Y": r"Transformer temperature $Y_t$ (°C)",
    "Ta": r"Outdoor temperature $T_{a,t}$ (°C)",
    "S": r"Solar radiation $\Phi_{s,t}$ (W/m²)",
    "I": r"Load $\Phi_{I,t}$ (kA)",
}
colors_map = {"Y": "firebrick", "Ta": "steelblue", "S": "orange", "I": "seagreen"}

fig, axes = plt.subplots(4, 1, figsize=(10, 8), sharex=True)
for ax, col in zip(axes, ["Y", "Ta", "S", "I"]):
    ax.plot(df["time"], df[col], color=colors_map[col], linewidth=0.7)
    ax.set_ylabel(var_labels[col], fontsize=9)
    ax.set_title(var_labels[col], fontsize=10)
axes[-1].set_xlabel("Time (hours)")
fig.suptitle("Transformer-Station Data", fontsize=13)
fig.tight_layout()
save_fig("image7")

## 2.2 — 1D state-space model

$$X_{t+1} = A\,X_t + B\,u_t + G\,e_{1,t}, \quad Y_t = C\,X_t + e_{2,t}$$

In [ ]:
def kf_general(A, B, C, Sig1, Sig2, X0, P0, Y, U, return_states=False):
    """
    General Kalman filter for n-dimensional state-space model.
    A: (n,n), B: (n,p), C: (m,n), Sig1: (n,n), Sig2: (m,m)
    X0: (n,1), P0: (n,n), Y: (T,m), U: (T,p)
    """
    n_ = A.shape[0]
    T_ = Y.shape[0]
    m_ = C.shape[0]

    x_est = X0.copy()
    P_est = P0.copy()
    loglik = 0.0
    y_pred_arr = np.zeros(T_)
    innov_arr = np.zeros(T_)
    S_arr = np.zeros(T_)
    x_pred_arr = np.zeros((T_, n_))
    x_filt_arr = np.zeros((T_, n_))

    for t_ in range(T_):
        u = U[t_].reshape(-1, 1)
        # prediction
        x_pred = A @ x_est + B @ u
        P_pred = A @ P_est @ A.T + Sig1
        x_pred_arr[t_] = x_pred.ravel()

        # innovation
        y_pred = C @ x_pred
        S_t = C @ P_pred @ C.T + Sig2
        inn = Y[t_].reshape(-1, 1) - y_pred
        y_pred_arr[t_] = y_pred.ravel()[0]
        innov_arr[t_] = inn.ravel()[0]
        S_arr[t_] = S_t.ravel()[0]

        # log-likelihood
        det_S = np.linalg.det(S_t)
        if det_S <= 0:
            return dict(loglik=-1e10)
        quad = inn.T @ np.linalg.solve(S_t, inn)
        loglik += -0.5 * (m_ * np.log(2 * np.pi) + np.log(det_S)
                          + quad.item())

        # update
        K = P_pred @ C.T @ np.linalg.inv(S_t)
        x_est = x_pred + K @ inn
        P_est = (np.eye(n_) - K @ C) @ P_pred
        x_filt_arr[t_] = x_est.ravel()

    out = dict(loglik=loglik, y_pred=y_pred_arr,
               innov=innov_arr, S=S_arr)
    if return_states:
        out["x_pred"] = x_pred_arr
        out["x_filt"] = x_filt_arr
    return out


def negll_1d(par, Y_arr, U_arr):
    """Negative log-likelihood for 1D model."""
    A_ = np.array([[par[0]]])
    B_ = np.array([[par[1], par[2], par[3]]])
    C_ = np.array([[1.0]])
    Sig1 = np.array([[np.exp(par[4])**2]])
    Sig2 = np.array([[np.exp(par[5])**2]])
    X0_ = np.array([[par[6]]])
    P0_ = np.array([[10.0]])
    try:
        res = kf_general(A_, B_, C_, Sig1, Sig2, X0_, P0_, Y_arr, U_arr)
        ll = res["loglik"]
    except Exception:
        return 1e10
    if not np.isfinite(ll):
        return 1e10
    return -ll


# Prepare data
Y_data = df[["Y"]].values   # (T, 1)
U_data = df[["Ta", "S", "I"]].values  # (T, 3)

# Fit 1D model
start_1d = [0.7, 0.1, 0.001, 0.3, np.log(0.5), np.log(0.1), df["Y"].iloc[0]]
bounds_1d = [(-1, 1), (-2, 2), (-1, 1), (-2, 2),
             (np.log(1e-4), np.log(50)), (np.log(1e-4), np.log(50)),
             (0, 60)]

print("Fitting 1D model ...")
res_1d = minimize(negll_1d, start_1d, args=(Y_data, U_data),
                  method="L-BFGS-B", bounds=bounds_1d,
                  options=dict(maxiter=2000))
p1d = res_1d.x
print(f"Convergence: {res_1d.success}, nll = {res_1d.fun:.3f}")

# Extract results at optimum
A1 = np.array([[p1d[0]]])
B1 = np.array([[p1d[1], p1d[2], p1d[3]]])
C1 = np.array([[1.0]])
Sig1_1d = np.array([[np.exp(p1d[4])**2]])
Sig2_1d = np.array([[np.exp(p1d[5])**2]])
X0_1d = np.array([[p1d[6]]])
P0_1d = np.array([[10.0]])

res1 = kf_general(A1, B1, C1, Sig1_1d, Sig2_1d, X0_1d, P0_1d,
                  Y_data, U_data, return_states=True)

loglik_1d = res1["loglik"]
k_1d = len(p1d)
n_obs = len(df)
aic_1d = -2 * loglik_1d + 2 * k_1d
bic_1d = -2 * loglik_1d + np.log(n_obs) * k_1d

print(f"\n=== 1D Model Parameters ===")
print(f"A      = {p1d[0]:.4f}")
print(f"B_Ta   = {p1d[1]:.4f}")
print(f"B_S    = {p1d[2]:.5f}")
print(f"B_I    = {p1d[3]:.4f}")
print(f"sigma1 = {np.exp(p1d[4]):.4f}")
print(f"sigma2 = {np.exp(p1d[5]):.4f}")
print(f"X0     = {p1d[6]:.4f}")
print(f"logLik = {loglik_1d:.3f}  AIC = {aic_1d:.2f}  BIC = {bic_1d:.2f}")

In [ ]:
# Plot 1D fit
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(df["time"], df["Y"], color="steelblue", linewidth=0.8, label="Observed")
ax.plot(df["time"], res1["y_pred"], "--", color="firebrick", linewidth=0.8,
        label="1-step prediction")
ax.set_xlabel("Time (hours)")
ax.set_ylabel("Temperature (°C)")
ax.set_title("1D State-Space Model: Observed vs. One-Step Prediction")
ax.legend()
save_fig("image8")

In [ ]:
# Residual diagnostics
def plot_diagnostics(resid, title, savename):
    fig, axes = plt.subplots(2, 2, figsize=(11, 7))
    fig.suptitle(title, fontsize=13)

    # Residuals vs time
    ax = axes[0, 0]
    ax.plot(resid, color="steelblue", linewidth=0.6)
    ax.axhline(0, color="red", ls="--", lw=0.8)
    ax.set_title("Residuals vs Time")
    ax.set_xlabel("Time (hours)")
    ax.set_ylabel("Residuals")

    # ACF
    ax = axes[0, 1]
    nlags = 30
    n_r = len(resid)
    r_mean = np.mean(resid)
    acf_vals = np.correlate(resid - r_mean, resid - r_mean, mode="full")
    acf_vals = acf_vals[n_r - 1:] / acf_vals[n_r - 1]
    acf_vals = acf_vals[:nlags + 1]
    ci = 1.96 / np.sqrt(n_r)
    ax.bar(range(nlags + 1), acf_vals, width=0.4, color="steelblue")
    ax.axhline(ci, color="red", ls="--", lw=0.8)
    ax.axhline(-ci, color="red", ls="--", lw=0.8)
    ax.set_title("Autocorrelation Function")
    ax.set_xlabel("Lag")
    ax.set_ylabel("ACF")

    # PACF (Levinson-Durbin)
    ax = axes[1, 0]
    from statsmodels.tsa.stattools import pacf as compute_pacf
    try:
        pacf_vals = compute_pacf(resid, nlags=nlags, method="ywm")
    except Exception:
        pacf_vals = np.zeros(nlags + 1)
    ax.bar(range(1, nlags + 1), pacf_vals[1:], width=0.4, color="steelblue")
    ax.axhline(ci, color="red", ls="--", lw=0.8)
    ax.axhline(-ci, color="red", ls="--", lw=0.8)
    ax.set_title("Partial Autocorrelation Function")
    ax.set_xlabel("Lag")
    ax.set_ylabel("PACF")

    # Q-Q plot
    ax = axes[1, 1]
    sorted_r = np.sort((resid - np.mean(resid)) / np.std(resid))
    theoretical = norm.ppf(np.linspace(0.5 / n_r, 1 - 0.5 / n_r, n_r))
    ax.scatter(theoretical, sorted_r, s=10, color="steelblue")
    lim = max(abs(theoretical.min()), abs(theoretical.max()))
    ax.plot([-lim, lim], [-lim, lim], "r-", lw=0.8)
    ax.set_title("Q-Q Plot")
    ax.set_xlabel("Theoretical Quantiles")
    ax.set_ylabel("Sample Quantiles")

    fig.tight_layout()
    save_fig(savename, fig)

resid_1d = res1["innov"]
plot_diagnostics(resid_1d, "1D Model: Residual Diagnostic Analysis", "image9")

rmse_1d = np.sqrt(np.mean(resid_1d**2))
print(f"RMSE 1D = {rmse_1d:.3f} °C")

## 2.3 — 2D state-space model

In [ ]:
def negll_2d(par, Y_arr, U_arr):
    """Negative log-likelihood for 2D model."""
    A_ = np.array(par[0:4]).reshape(2, 2)
    B_ = np.array(par[4:10]).reshape(2, 3)
    C_ = np.array([[1.0, 0.0]])
    L = np.array([[par[10], 0.0],
                  [par[11], par[12]]])
    Sig1 = L @ L.T
    Sig2 = np.array([[np.exp(par[13])**2]])
    X0_ = np.array(par[14:16]).reshape(2, 1)
    P0_ = np.eye(2) * 10.0
    try:
        res = kf_general(A_, B_, C_, Sig1, Sig2, X0_, P0_, Y_arr, U_arr)
        ll = res["loglik"]
    except Exception:
        return 1e10
    if not np.isfinite(ll):
        return 1e10
    return -ll


start_2d = [
    0.7, 0.3, 0.01, 0.2,                    # A (row-major)
    0.1, 0.001, 0.3, 0.1, 0.0, 0.1,         # B (row-major)
    0.5, 0.1, 0.5,                           # L11, L21, L22
    np.log(0.1),                             # log sigma2
    df["Y"].iloc[0], df["Y"].iloc[0] - 5,    # X0
]

bounds_2d = (
    [(-2, 2)] * 4 +       # A
    [(-2, 2)] * 6 +       # B
    [(-3, 3)] * 3 +       # L
    [(np.log(1e-4), np.log(50))] +  # log sigma2
    [(-50, 60)] * 2        # X0
)

print("Fitting 2D model ...")
res_2d = minimize(negll_2d, start_2d, args=(Y_data, U_data),
                  method="L-BFGS-B", bounds=bounds_2d,
                  options=dict(maxiter=3000))
p2d = res_2d.x
print(f"Convergence: {res_2d.success}, nll = {res_2d.fun:.3f}")

# Extract at optimum
A2 = p2d[0:4].reshape(2, 2)
B2 = p2d[4:10].reshape(2, 3)
C2 = np.array([[1.0, 0.0]])
L2 = np.array([[p2d[10], 0.0], [p2d[11], p2d[12]]])
Sig1_2d = L2 @ L2.T
Sig2_2d = np.array([[np.exp(p2d[13])**2]])
X0_2d = p2d[14:16].reshape(2, 1)
P0_2d = np.eye(2) * 10.0

res2 = kf_general(A2, B2, C2, Sig1_2d, Sig2_2d, X0_2d, P0_2d,
                  Y_data, U_data, return_states=True)

loglik_2d = res2["loglik"]
k_2d = len(p2d)
aic_2d = -2 * loglik_2d + 2 * k_2d
bic_2d = -2 * loglik_2d + np.log(n_obs) * k_2d

print(f"\n=== 2D Model Parameters ===")
print(f"A =\n{np.array2string(A2, precision=4)}")
print(f"B =\n{np.array2string(B2, precision=4)}")
print(f"Sigma1 =\n{np.array2string(Sig1_2d, precision=4)}")
print(f"sigma2 = {np.exp(p2d[13]):.5f}")
print(f"X0 = {X0_2d.ravel()}")
print(f"logLik = {loglik_2d:.3f}  AIC = {aic_2d:.2f}  BIC = {bic_2d:.2f}")

In [ ]:
# Plot 2D fit
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(df["time"], df["Y"], color="steelblue", linewidth=0.8, label="Observed")
ax.plot(df["time"], res2["y_pred"], "--", color="firebrick", linewidth=0.8,
        label="1-step prediction")
ax.set_xlabel("Time (hours)")
ax.set_ylabel("Temperature (°C)")
ax.set_title("2D State-Space Model: Observed vs. One-Step Prediction")
ax.legend()
save_fig("image10")

In [ ]:
# Diagnostics 2D
resid_2d = res2["innov"]
plot_diagnostics(resid_2d, "2D Model: Residual Diagnostic Analysis", "image11")

rmse_2d = np.sqrt(np.mean(resid_2d**2))
print(f"RMSE 2D = {rmse_2d:.3f} °C")

In [ ]:
# AIC / BIC comparison
fig, axes = plt.subplots(1, 2, figsize=(9, 4.5))
for ax, crit, vals in zip(axes, ["AIC", "BIC"],
                          [(aic_1d, aic_2d), (bic_1d, bic_2d)]):
    bars = ax.bar(["1D", "2D"], vals, width=0.5,
                  color=["steelblue", "firebrick"])
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, v + 3,
                f"{v:.1f}", ha="center", fontsize=11)
    ax.set_title(crit, fontsize=12)
    ax.set_ylabel("Criterion value")
    ax.set_ylim(0, max(vals) * 1.12)
fig.suptitle("Model Comparison: Information Criteria", fontsize=13)
fig.tight_layout()
save_fig("image12")

## 2.4 — Interpretation of the two reconstructed states

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(11, 8), sharex=True)

panels = [
    (axes[0, 0], res2["x_filt"][:, 0], "Filtered State X1 (°C)", "firebrick"),
    (axes[0, 1], res2["x_filt"][:, 1], "Filtered State X2 (°C)", "firebrick"),
    (axes[1, 0], df["I"].values, "Load Current ΦI (kA)", "steelblue"),
    (axes[1, 1], df["Y"].values, "Observed Temperature Y (°C)", "black"),
    (axes[2, 0], df["Ta"].values, "Outdoor Temperature Ta (°C)", "steelblue"),
    (axes[2, 1], df["S"].values, "Solar Radiation Φs (W/m²)", "steelblue"),
]
for ax, data, title, col in panels:
    ax.plot(df["time"], data, color=col, linewidth=0.6)
    ax.set_title(title, fontsize=10)
axes[2, 0].set_xlabel("Time (hours)")
axes[2, 1].set_xlabel("Time (hours)")
fig.suptitle("2D State-Space Model: Latent States and System Inputs", fontsize=13)
fig.tight_layout()
save_fig("image13")

In [ ]:
print("\n=== Final Summary ===")
print(f"1D: logLik={loglik_1d:.2f}, AIC={aic_1d:.2f}, BIC={bic_1d:.2f}, RMSE={rmse_1d:.3f}")
print(f"2D: logLik={loglik_2d:.2f}, AIC={aic_2d:.2f}, BIC={bic_2d:.2f}, RMSE={rmse_2d:.3f}")